# MedAI — Qwen2.5 + QLoRA + RAG (FAISS) + Hugging Face Spaces

End-to-end pipeline that follows the project PDFs:

**Training:** MedQuAD → cleaning → chat template → train/val/test → Qwen tokenizer → 4-bit frozen Qwen + LoRA → cross-entropy (answer tokens only) → backprop → 8-bit AdamW → trained adapter
**Evaluation:** ROUGE-L, BERTScore, eval loss, perplexity — base vs fine-tuned vs fine-tuned + RAG
**RAG:** answers → chunks → bge-small embeddings → FAISS → top-K → question + context → Qwen + LoRA
**Deployment:** adapter → Hugging Face Hub, app + FAISS index → Hugging Face Space (Gradio)

### Before you run
1. Runtime: **GPU (T4 or better)**. Kaggle (free 2×T4, 12 h sessions) is more reliable than free Colab for the 7B model.
2. Push this repository to GitHub (public) and paste its URL in the config cell.
3. Create a Hugging Face **write** token: https://huggingface.co/settings/tokens

### Why the original notebook was so slow (and what changed)
| Original | Now | Effect |
|---|---|---|
| `padding="max_length"`, `max_length=2048` | dynamic padding, `MAX_LEN=512` | most examples are ~250–450 tokens, so ~4–6× less compute |
| 3 epochs × 13,087 examples, batch 1 | 1 epoch, configurable subset, batch 4 × accum 4 | ~10× fewer optimizer steps |
| full 1,636-example eval every 100 steps | 200-example eval | eval no longer dominates |
| `pad_token = eos_token`, labels = inputs | Qwen's own pad token, loss on answer tokens only | model learns to stop; no loss on the prompt |
| custom `### Instruction` text | Qwen chat template (as in the PDF) | matches how the instruct model was trained |
| default AdamW | `paged_adamw_8bit` | the 8-bit AdamW from the PDF, less memory |

## 0. Setup

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets sentence-transformers faiss-cpu evaluate rouge_score bert_score huggingface_hub matplotlib

In [ ]:
# ---------------- CONFIG — edit these ----------------
GITHUB_REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/medai.git"
HF_USERNAME     = "YOUR-HF-USERNAME"

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"     # "Qwen/Qwen2.5-1.5B-Instruct" -> ~4-5x faster run
MAX_LEN = 512                # tokens per training example (prompt + answer)
MAX_TRAIN_SAMPLES = 4000     # None = all ~13k training examples (roughly 3x longer)
NUM_EPOCHS = 1
BATCH_SIZE, GRAD_ACCUM = 4, 4            # effective batch 16
LEARNING_RATE = 2e-4
N_VAL = 200                  # validation examples used for eval loss during training
N_EVAL = 50                  # test questions for ROUGE-L / BERTScore generation eval

model_short  = MODEL_NAME.split("/")[-1].lower().replace("-instruct", "")
ADAPTER_REPO = f"{HF_USERNAME}/{model_short}-medquad-qlora"
SPACE_REPO   = f"{HF_USERNAME}/MedAI"
OUTPUT_DIR   = "qwen-medical-qlora"
INDEX_DIR    = "rag_index"

In [ ]:
import os, sys
repo_dir = GITHUB_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not os.path.exists(repo_dir):
    !git clone {GITHUB_REPO_URL}
sys.path.insert(0, os.path.join(repo_dir, "space"))
import rag      # shared cleaning / chunking / FAISS / prompt code (same file the Space uses)

from huggingface_hub import login
login()         # paste your HF *write* token (on Kaggle you can also use Secrets)

## 1–4. Load MedQuAD, clean, split  (PDF Steps 1–4)

In [ ]:
from datasets import load_dataset
raw = load_dataset(rag.DATASET_ID)["train"].to_pandas()
print("Raw:", raw.shape, "| nulls:", raw[["Question","Answer"]].isnull().sum().to_dict())

df = rag.clean_df(raw)                  # dropna, strip, remove empty, de-duplicate
train_df, val_df, test_df = rag.split_df(df)
print("Clean:", len(df), "| train", len(train_df), "| val", len(val_df), "| test", len(test_df))

df["answer_words"] = df["Answer"].str.split().str.len()
print(df["answer_words"].describe().round(1))

## 5. Tokenizer + Qwen chat template  (PDF Steps 3, 5)

In [ ]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("pad:", tokenizer.pad_token, "| eos:", tokenizer.eos_token)   # Qwen has its own pad token — don't overwrite it

def to_messages(q, a=None):
    m = [{"role": "system", "content": rag.SYSTEM_PROMPT}, {"role": "user", "content": q}]
    if a is not None:
        m.append({"role": "assistant", "content": a})
    return m

print(tokenizer.apply_chat_template(to_messages(train_df.Question[0], train_df.Answer[0]), tokenize=False)[:900])

In [ ]:
from datasets import Dataset

def tokenize_example(ex):
    # Loss only on the answer: prompt positions get label -100 (ignored by cross-entropy)
    prompt_txt = tokenizer.apply_chat_template(to_messages(ex["Question"]), tokenize=False, add_generation_prompt=True)
    full_txt   = tokenizer.apply_chat_template(to_messages(ex["Question"], ex["Answer"]), tokenize=False)
    p_ids = tokenizer(prompt_txt, add_special_tokens=False)["input_ids"]
    a_ids = tokenizer(full_txt[len(prompt_txt):], add_special_tokens=False)["input_ids"]  # answer + <|im_end|>
    ids    = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_src = train_df.sample(MAX_TRAIN_SAMPLES, random_state=42) if MAX_TRAIN_SAMPLES else train_df
val_src   = val_df.sample(min(N_VAL, len(val_df)), random_state=42)
cols = ["Question", "Answer"]
tok_train = Dataset.from_pandas(train_src[cols].reset_index(drop=True)).map(tokenize_example, remove_columns=cols)
tok_val   = Dataset.from_pandas(val_src[cols].reset_index(drop=True)).map(tokenize_example, remove_columns=cols)

import numpy as np
lens = np.array([len(x) for x in tok_train["input_ids"]])
print(f"train examples: {len(tok_train)} | token length mean {lens.mean():.0f}, median {np.median(lens):.0f}, "
      f"truncated at {MAX_LEN}: {(lens >= MAX_LEN).mean():.1%}")
print(f"Padding everything to 2048 (old notebook) would have cost {2048 / lens.mean():.1f}x more tokens.")

## 6–12. Load Qwen in 4-bit (NF4) + attach LoRA = QLoRA  (PDF Steps 6–12)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

use_bf16 = torch.cuda.is_bf16_supported()          # False on T4 -> fp16
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, dtype=compute_dtype,
    device_map={"": 0}, attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()      # base weights frozen, only LoRA A/B train

## 13–19. Train: forward → cross-entropy → backprop → 8-bit AdamW → LoRA update

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    optim="paged_adamw_8bit",            # 8-bit AdamW optimizer states
    bf16=use_bf16, fp16=not use_bf16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50, save_total_limit=2,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=tok_train, eval_dataset=tok_val, data_collator=collator)
steps = len(tok_train) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM)
print(f"~{steps} optimizer steps — watch the progress bar ETA after the first few steps")

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
import math, matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), "o-", label="val loss")
plt.xlabel("step"); plt.ylabel("cross-entropy"); plt.legend(); plt.title("QLoRA fine-tuning on MedQuAD")
plt.savefig("loss_curve.png", dpi=120, bbox_inches="tight"); plt.show()

eval_metrics = trainer.evaluate()
val_loss = eval_metrics["eval_loss"]
print(f"Validation loss {val_loss:.4f} | perplexity {math.exp(val_loss):.2f}")

In [ ]:
# Save + publish the adapter (only LoRA weights, not a second copy of the 7B model)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model.push_to_hub(ADAPTER_REPO)
tokenizer.push_to_hub(ADAPTER_REPO)
print("Adapter published: https://huggingface.co/" + ADAPTER_REPO)

## 20–26. Build the RAG index: chunks → embeddings → FAISS

In [ ]:
embedder = rag.load_embedder(device="cuda")
chunks = rag.build_chunks(train_df, exclude_answers=set(test_df["Answer"]))  # no test answers in the index
index = rag.build_index(chunks, embedder, INDEX_DIR)
retriever = rag.Retriever(INDEX_DIR, embedder=embedder)
print("chunks indexed:", index.ntotal)

for c in retriever.search("What are the common causes of anemia?", k=3):
    print(f"{c['score']:.3f} | {c['source_question']}")

## Evaluation: base vs fine-tuned vs fine-tuned + RAG  (ROUGE-L, BERTScore)

In [ ]:
import evaluate, pandas as pd

model.eval(); model.config.use_cache = True
tokenizer.padding_side = "left"          # left-pad for batched generation

@torch.no_grad()
def generate_batch(msg_lists, max_new_tokens=256, bs=8):
    outs = []
    for i in range(0, len(msg_lists), bs):
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msg_lists[i:i+bs]]
        enc = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                             repetition_penalty=1.05, pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

eval_df = test_df.sample(N_EVAL, random_state=0).reset_index(drop=True)
qs, refs = eval_df["Question"].tolist(), eval_df["Answer"].tolist()

plain_msgs = [rag.build_messages(q) for q in qs]
rag_msgs   = [rag.build_messages(q, retriever.search(q, k=4)) for q in qs]

with model.disable_adapter():                         # original Qwen, no LoRA
    preds_base = generate_batch(plain_msgs)
preds_ft  = generate_batch(plain_msgs)                # fine-tuned
preds_rag = generate_batch(rag_msgs)                  # fine-tuned + RAG

In [ ]:
rouge, bertscore = evaluate.load("rouge"), evaluate.load("bertscore")
def score(preds):
    r = rouge.compute(predictions=preds, references=refs)["rougeL"]
    b = bertscore.compute(predictions=preds, references=refs, lang="en", model_type="distilbert-base-uncased")["f1"]
    return {"ROUGE-L": round(r, 4), "BERTScore-F1": round(float(np.mean(b)), 4)}

results = pd.DataFrame({
    "Qwen base":           score(preds_base),
    "Qwen + QLoRA":        score(preds_ft),
    "Qwen + QLoRA + RAG":  score(preds_rag),
}).T
results["val_loss (FT)"] = round(val_loss, 4); results["perplexity (FT)"] = round(math.exp(val_loss), 2)
print(f"n = {N_EVAL} held-out test questions")
display(results)
results.to_csv("eval_results.csv")

pd.set_option("display.max_colwidth", 300)
display(pd.DataFrame({"question": qs[:3], "reference": refs[:3], "fine-tuned+RAG": preds_rag[:3]}))

## Deploy to Hugging Face Spaces
Uploads `space/` (app.py, rag.py, requirements.txt, README.md) plus the FAISS index and sets the Space variables.
**Hardware:** a 7B model needs a GPU. Free accounts in good standing can host a limited number of **ZeroGPU** Spaces; otherwise pick ZeroGPU/GPU in the Space *Settings*. On the free **CPU basic** tier use the 1.5B or 0.5B model instead (set `MODEL_NAME` above and retrain).

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
try:
    api.create_repo(SPACE_REPO, repo_type="space", space_sdk="gradio", space_hardware="zero-a10g", exist_ok=True)
except Exception as e:
    print("Could not request ZeroGPU automatically (", e, ") — creating on default hardware; change it in Settings.")
    api.create_repo(SPACE_REPO, repo_type="space", space_sdk="gradio", exist_ok=True)

api.upload_folder(folder_path=os.path.join(repo_dir, "space"), repo_id=SPACE_REPO, repo_type="space")
api.upload_folder(folder_path=INDEX_DIR, path_in_repo="rag_index", repo_id=SPACE_REPO, repo_type="space")
api.add_space_variable(SPACE_REPO, "BASE_MODEL", MODEL_NAME)
api.add_space_variable(SPACE_REPO, "ADAPTER_ID", ADAPTER_REPO)
print("Space: https://huggingface.co/spaces/" + SPACE_REPO, "— first build downloads the model, allow ~5-10 min.")

## Project summary (copy the real numbers into your README / resume)

In [ ]:
r = results
print(f'''Fine-tuned {MODEL_NAME} with QLoRA (4-bit NF4, LoRA r=16) on {len(tok_train):,} MedQuAD QA pairs;
val loss {val_loss:.3f}, perplexity {math.exp(val_loss):.2f}.
On {N_EVAL} held-out questions: ROUGE-L {r.loc["Qwen base","ROUGE-L"]:.3f} (base) -> {r.loc["Qwen + QLoRA","ROUGE-L"]:.3f} (QLoRA) -> {r.loc["Qwen + QLoRA + RAG","ROUGE-L"]:.3f} (QLoRA+RAG);
BERTScore-F1 {r.loc["Qwen base","BERTScore-F1"]:.3f} -> {r.loc["Qwen + QLoRA","BERTScore-F1"]:.3f} -> {r.loc["Qwen + QLoRA + RAG","BERTScore-F1"]:.3f}.
RAG: {index.ntotal:,} chunks, bge-small-en-v1.5 embeddings, FAISS IndexFlatIP (cosine), top-4.
Demo: https://huggingface.co/spaces/{SPACE_REPO}''')